# Proxy Clinical: Block 2 pilot launcher

Thin launcher. No training logic lives here; everything runs from the repo.
Runtime: **A100 (high-RAM)**. Re-run cell 5 with the same `RUN_ID` to resume after a disconnect.

Before running: upload `proxy-clinical.bundle` (or the tarball) and the pilot JSONL
(`corpus.jsonl`, `train.jsonl`, `val.jsonl`, `meta.json`) to `MyDrive/proxy-clinical/` and
`MyDrive/proxy-clinical/data/pilot/`. Add `HF_TOKEN` in the Secrets panel (needed only for gated bases such as Llama).

Order on a fresh runtime: 1, 2, 3, then **Runtime > Restart session** (pip replaced Colab's preinstalled
stack), then 1, 2, 4, 5. Cell 3 is skipped after the restart.


In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/proxy-clinical'
import os; assert os.path.isdir(DRIVE), f'{DRIVE} missing: upload the bundle and data first'


In [ ]:
# 2. Pull the repo from the bundle on Drive (source of truth is the repo, not this notebook)
import os, subprocess
REPO = '/content/proxy-clinical'
if not os.path.isdir(REPO):
    if os.path.exists(f'{DRIVE}/proxy-clinical.bundle'):
        subprocess.run(['git', 'clone', f'{DRIVE}/proxy-clinical.bundle', REPO], check=True)
    else:
        subprocess.run(['bash', '-c', f'mkdir -p {REPO} && tar -xzf {DRIVE}/proxy-clinical-block2.tar.gz -C /content'], check=True)
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)


In [ ]:
# 2. Pinned environment (replaces Colab's preinstalled stack). Then: Runtime > Restart session, then run 1, 2, 4, 5.
# Colab's preinstalled torchvision/torchaudio are built against Colab's torch; after torch is replaced they fail to
# load and transformers surfaces that as a bogus import error. Nothing here needs them, so they are removed.
%pip uninstall -y -q torchvision torchaudio
%pip install -q -r requirements-colab.txt
import subprocess, sys
out = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())'],
                     capture_output=True, text=True).stdout.strip()
print('torch after install:', out)
if out.endswith('False'):
    # Driver older than the CUDA 13 wheel needs: same torch version, CUDA 12.6 build.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'torch==2.14.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
    print('reinstalled torch (cu126 build)')
chk = subprocess.run([sys.executable, '-c', 'import transformers, trl, peft; from transformers import AutoModelForCausalLM; print("imports ok")'],
                     capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr.strip()[-800:])
print('Now: Runtime > Restart session, then run cells 1, 2, 4, 5.')


In [ ]:
# 4. Data from Drive + HF token from Secrets (never in a cell)
import os, shutil, hashlib, glob
os.makedirs('data/pilot', exist_ok=True)
for f in glob.glob(f'{DRIVE}/data/pilot/*.json*'):
    shutil.copy(f, 'data/pilot/')
for name in ('corpus.jsonl', 'train.jsonl', 'val.jsonl'):
    p = f'data/pilot/{name}'
    assert os.path.exists(p), f'missing {p}'
    print(name, hashlib.sha256(open(p, 'rb').read()).hexdigest()[:16], os.path.getsize(p), 'bytes')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Secrets')
except Exception as e:
    print('HF_TOKEN not set (fine for Qwen2.5, required for Llama):', type(e).__name__)


In [ ]:
# 5. Run the pilot: train -> infer -> evaluate -> determinism. Artifacts land in one versioned folder on Drive.
import datetime, os
RUN_ID = os.environ.get('RUN_ID') or f"pilot-qwen2.5-3b-{datetime.date.today():%Y%m%d}"   # keep the same RUN_ID to resume
OUT = f'{DRIVE}/runs/{RUN_ID}'
print('run folder:', OUT)
!bash scripts/run_pilot.sh configs/pilot.yaml "$OUT"


Artifacts in the run folder: `adapter/` (LoRA weights + tokenizer), `run_manifest.json`, `config.yaml`,
`predictions.jsonl` (+ manifest), `eval_report.md`, `eval.json`, `determinism.json`, `pip_freeze.txt`, checkpoints.

The determinism claim proven above is: same checkpoint, same inputs, same pinned environment, same session,
byte-identical output. It is not a cross-GPU claim.
